# 모델 개발 전 데이터 셋 정리 과정
1. 중복된 화합물 개수 찾아내기
2. 중복된 화합물 중 label 정보가 일치하지 않는 화합물 찾기
3. smiles 코드가 없는 물질 제거하기
4. 최종 데이터 확인 (label의 밸런스 확인)

In [1]:
import pandas as pd

df = pd.read_excel('skin_irritation.xlsx', sheet_name=2)

In [2]:
df.columns

Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Concentration', 'Concentration_Units', 'Mixture', 'Species',
       'Reported_Strain', 'Strain', 'Sex', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'SMILES', 'Preferred_Name', 'Synonyms',
       'URL_CompTox', 'URL_CEBS'],
      dtype='str')

In [3]:
df_selec = df[(df['Mixture']=='Chemical') & (df['Species']=='Human')] #Mixture 컬럼이 Chemical, Species 컬럼은 Human인 경우 선택.

In [4]:
df_selec['Endpoint'].value_counts()

Endpoint
Qualitative classification    90
Positive reaction             90
Name: count, dtype: int64

In [5]:
df_class = df_selec[df_selec['Endpoint']=='Qualitative classification']

In [6]:
df_react = df_selec[df_selec['Endpoint']=='Positive reaction']

In [7]:
df_class['Response'].value_counts()

Response
Not classified                      65
Irritant                            14
Not classified/Possible irritant     8
Irritant/Possible corrosive          3
Name: count, dtype: int64

In [8]:
# Qualitatie classification 정리
df_class['label']=0  #label을 0으로 먼저 초기화하고,,,
df_class.loc[df_class['Response']!='Not classified','label']=1  #Not classified가 아니라면 (!=) label을 1로 설정.

In [9]:
# Positive reaction 정리
df_react['label']=0
df_react.loc[df_react['Response']!='0','label']=1 #Respons 값이 '0'이 아니라면 (!=) label을 1로 설정.

# Quiz1
왜 label을 정의할 때, Response 컬럼을 굳이 numerical value로 바꾸지 않아도 될까요?

In [10]:
# 일치여부만 따지는 거라서

In [11]:
df_total = pd.concat([df_class, df_react])
df_total['label'].value_counts()

label
0    102
1     78
Name: count, dtype: int64

# 데이터 중복 문제
1. 180개 데이터가 있지만, 화합물 개수는 중복되어 있을 수 있으니 확인 필요.
2. 중복된 화합물 중에서 label이 불일치 하는 경우가 있는지 확인 필요.

In [12]:
df_total['Chemical_Name']

0                  Alcohol ethoxylate C12-15/E5 phosphate
1           C12-13 beta-branched primary alco hol sulfate
2       C12-13 beta-branched primary alco hol sulfate/...
5                   N,N-Dimethyl-N-dodecyl amino- betaine
6       Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)],...
                              ...                        
1850                                      Triethanolamine
1852                                        Decanoic acid
1858                                       1-Tetradecanol
1860       Ethylenediaminetetraacetic acid, disodium salt
1862               Hexadecanoic acid, 1-methylethyl ester
Name: Chemical_Name, Length: 180, dtype: str

In [13]:
df_total['Chemical_Name'].to_list()

['Alcohol ethoxylate C12-15/E5 phosphate',
 'C12-13 beta-branched primary alco hol sulfate',
 'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
 'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
 'Alkyl polyglucoside 600',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C11/E3',
 'Tween 80',
 'Tween 80',
 'Propylene glycol',
 'Heptanal',
 'Isopropyl myristate',
 'di-Propylene glycol',
 'Sodium hydroxide',
 'Sodium hydroxide',
 '4-Methylthio benzaldehyde',
 'Heptyl butyrate',
 'Hydroxycitronellal',
 'Methyl caproate',
 'Alkyl dimethyl betaine',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 'Benzyl salicylate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium carbonate',
 'Benzalkonium chloride',
 '1-Bro

In [14]:
chemical_name = df_total['Chemical_Name'].to_list()
unique_name = set(chemical_name) #set()을 사용하면, 중복된 item은 모두 삭제됩니다.

In [15]:
unique_name

{'(2E)-3,7-Dimethyl-2,6-octadien-1-ol',
 '1,2-Propylene glycol',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 '1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)',
 '1-Bromo-4-chlorobutane',
 '1-Bromohexane',
 '1-Decanol',
 '1-Dodecanol',
 '1-Hexanol',
 '1-Naphthalene acetic acid',
 '1-Naphthaleneacetic acid',
 '1-Octanol',
 '1-Pentanol',
 '1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one',
 '1-Tetradecanol',
 '1-tert-Butoxy-2-propanol',
 '10-Undecenoic acid',
 '2-Isopropyl-2-isobutyl-1,3-dime- thoxypropane',
 '3,4-Dimethyl-1H-pyrazole',
 '4-(Methylthio)benzaldehyde',
 '4-Methylthio benzaldehyde',
 'Acetic acid',
 'Alcohol ethoxylate C11/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C12-15/E5 phosphate',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alkyl dimethyl betaine',
 'Alkyl polyglucoside 600',
 'Amines, hydrogenated tallow alkyl',
 'Benzalkonium chloride',
 'Benzene, 1-(1-methylethyl)-2-

In [16]:
len(unique_name) #180개 데이터가 있었지만, 단일 물질 기준으로는 119개가 있네요.

119

In [17]:
for each in unique_name:
    print (each)

Hexyl salicylate
Hexanol
Isopropanol
Isopropyl tetradecanoate
1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)
Tromethamine
Butan-1-ol
Benzyl alcohol
Fatty acids, potassium salts
1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one
1-Tetradecanol
Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex
Polyethylene glycol 400
Alkyl dimethyl betaine
alpha-Terpinyl acetate
Cocotrimethyl ammonium chloride
Carbonic acid sodium salt (1:2)
Heptanal
Decanol
Dodecanoic acid
3,4-Dimethyl-1H-pyrazole
Dipropyl disulfide
Methyl caproate
1-Naphthalene acetic acid
Amines, hydrogenated tallow alkyl
Hexadecanoic acid, 1-methylethyl ester
Sodium perborate
Hydroxycitronellal
1-Bromo-4-chlorobutane
Ethanol
1,2-Propylene glycol
Methyl laurate
Methyl dodecanoate
Heptyl butyrate
Eugenol
1-Hexanol
1-Dodecanol
Ethylenediaminetetraacetic acid, disodium salt
Benzyl-C8-18-alkyldimethylammonium chlorides
di-Propylene glycol
Tris(hydroxymethyl)aminomethane
Sodium carbonate
N,N-Dimethyl-N-dodecyl amino- beta

In [18]:
#label 정보는 동일 화합물이라면 1이거나 0이어야 함. nunique()는 동일 화합물의 label에 있는 정보의 개수 확인.
#groupby(Chemical Name)은 화합물 이름을 기준으로 dataframe을 묶어봅니다.
#이름 기준으로 묶었을 때, label 정보가 몇개 있는지 확인하는 방식.
df_total.groupby('Chemical_Name')['label'].nunique()

Chemical_Name
(2E)-3,7-Dimethyl-2,6-octadien-1-ol                                 1
1,2-Propylene glycol                                                1
1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)         1
1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)    1
1-Bromo-4-chlorobutane                                              1
                                                                   ..
alpha-Terpineol                                                     1
alpha-Terpinyl acetate                                              1
di-Propylene glycol                                                 1
di-n-Propyl disulfide                                               1
n-Pentanol                                                          1
Name: label, Length: 119, dtype: int64

# Quiz2
df_total.groupby('Chemical_Name')['label'].nunique()를 실행했을 때, 도대체 왜 최대값은 2, 최소값은 1이 나올까요?

In [19]:
# 라벨이라는 컬럼을 제대로 만들었으면 그 안에 0, 1밖에 없음

In [20]:
df_total.groupby('Chemical_Name')['label'].nunique().max()

np.int64(2)

In [21]:
df_total.groupby('Chemical_Name')['label'].nunique().min()

np.int64(1)

In [22]:
(df_total.groupby('Chemical_Name')['label'].nunique() == 2).mean()

np.float64(0.13445378151260504)

In [23]:
label_counts = df_total.groupby('Chemical_Name')['label'].nunique()
bad_names = label_counts[label_counts > 1].index
print (bad_names)
print ('label 정보가 불일치하는 화합물 개수: ',len(bad_names))

Index(['10-Undecenoic acid', 'Acetic acid', 'Alcohol ethoxylate C11/E3',
       'Alcohol ethoxylate C12-15/E5 phosphate', 'Alkyl polyglucoside 600',
       'Benzyl alcohol', 'Dodecanoic acid', 'Ethanol', 'Eugenol', 'Hexanol',
       'Hydrochloric acid', 'Linalyl acetate', 'Octanol', 'Sodium perborate',
       'Tween 80', 'Water'],
      dtype='str', name='Chemical_Name')
label 정보가 불일치하는 화합물 개수:  16


In [24]:
df_filtered = df_total.groupby('Chemical_Name').filter(lambda x: x['label'].nunique() == 1)

print(f"전체: {df_total.shape}행")
print(f"label이 불일치 하는 화합물 제거: {df_filtered.shape}행")

전체: (180, 26)행
label이 불일치 하는 화합물 제거: (143, 26)행


In [25]:
df_cleaned = df_filtered.drop_duplicates(subset='Chemical_Name')
print(f"label이 일치 하는 화합물 중 중복된 결과 제거: {df_cleaned.shape}행")

label이 일치 하는 화합물 중 중복된 결과 제거: (103, 26)행


# Quiz3
1. smiles 코드가 없는 물질 이름을 확인해봅시다.
2. pubchem 데이터베이스 (https://pubchem.ncbi.nlm.nih.gov/)에서 물질 명을 입력해서 검색해봅시다.
3. 구글에서 물질 명을 입력해서 검색해봅시다.
4. CASRN (카스넘버)는 물질의 이름을 표시하는 또 다른 방법입니다. 동일 화학물질이 여러개의 이름을 가질 수 있어서, 고유한 번호를 부여해서 화합물을 찾아요.
5. df_cleaned에서 smiles 코드가 없는 물질들은 왜 없는지 검색을 통해 찾아봅시다.

In [26]:
# 단일물질이 아니라 사슬로 결정됨 사슬의 길이를 결정할 수 없음

In [27]:
#smiles 코드가 없는 물질?
df_cleaned['SMILES']

1                                                     NaN
2                                                     NaN
5                                                     NaN
6                                                     NaN
8                                                     NaN
                              ...                        
1757                               COCC(COC)(CC(C)C)C(C)C
1764                          C=CCCC(=O)C1=CCCC2(CCCC2)C1
1858                                      CCCCCCCCCCCCCCO
1860    [Na+].[Na+].OC(=O)CN(CCN(CC(O)=O)CC([O-])=O)CC...
1862                           CCCCCCCCCCCCCCCC(=O)OC(C)C
Name: SMILES, Length: 103, dtype: str

In [28]:
df_cleaned['SMILES'].isna()

1        True
2        True
5        True
6        True
8        True
        ...  
1757    False
1764    False
1858    False
1860    False
1862    False
Name: SMILES, Length: 103, dtype: bool

In [29]:
df_smi = df_cleaned.dropna(subset=['SMILES'])

In [30]:
df_smi.shape

(81, 26)

In [31]:
df_smi['label'].value_counts()

label
0    56
1    25
Name: count, dtype: int64

In [32]:
df_smi.to_csv('skin_irritation_cleaned.csv', index=False)

print('저장 완료!')

저장 완료!
